In [1]:
import os
import pandas as pd
import numpy as np
import joblib

# Anchor to project root (same pattern as before)
os.chdir('/Users/eldigardonaz/Documents/WorldCup-2026-Analytics')

# Load the saved model bundle from Phase 4
bundle = joblib.load('models/wc2026_xgb_model.pkl')
model = bundle['model']
le = bundle['label_encoder']
feature_cols = bundle['feature_cols']

print("Model loaded ✓")
print("Model type:", type(model).__name__)
print("Label mapping:", dict(enumerate(le.classes_)))
print("Features expected:", len(feature_cols))
print("Weighting strength:", bundle['weighting_strength'])
print("Test accuracy:", bundle['test_accuracy'])

# Load the 2026 data and Elo for building match features
matches_2026 = pd.read_csv('data/raw/wc2026/matches_detailed.csv')
elo = pd.read_csv('data/raw/wc2026/elo_ratings_wc2026.csv')

print("\n2026 matches:", len(matches_2026))
print("Stages present:", matches_2026['stage_name'].unique())

Model loaded ✓
Model type: XGBClassifier
Label mapping: {0: 'Away Win', 1: 'Draw', 2: 'Home Win'}
Features expected: 14
Weighting strength: 0.65
Test accuracy: 0.462

2026 matches: 72
Stages present: <StringArray>
['Group Stage']
Length: 1, dtype: str


In [2]:
# --- Build match features for any two teams ---
# Uses the most recent Elo rating for each team

def get_team_elo(team_name, elo_df, year=2026):
    """Get a team's most recent Elo rating."""
    team_rows = elo_df[(elo_df['country'] == team_name) & (elo_df['year'] == year)]
    if len(team_rows) == 0:
        return None
    # Use the most recent snapshot
    return team_rows.sort_values('snapshot_date')['rating'].iloc[-1]

# Quick test
print("Argentina Elo:", get_team_elo('Argentina', elo))
print("Brazil Elo:", get_team_elo('Brazil', elo))
print("France Elo:", get_team_elo('France', elo))

Argentina Elo: 2113
Brazil Elo: 1984
France Elo: 2081


In [3]:
# --- Build features for a matchup and predict probabilities ---

def get_team_form(team_name, results_df):
    """Get a team's recent form (last 5 matches) from historical data.
       Returns avg points, goals for, goals against."""
    # Matches where this team played (home or away)
    home = results_df[results_df['home_team'] == team_name].copy()
    away = results_df[results_df['away_team'] == team_name].copy()
    
    # Build unified perspective
    home_p = pd.DataFrame({
        'date': home['date'], 'gf': home['home_score'], 'ga': home['away_score']
    })
    away_p = pd.DataFrame({
        'date': away['date'], 'gf': away['away_score'], 'ga': away['home_score']
    })
    log = pd.concat([home_p, away_p]).sort_values('date').tail(5)
    
    if len(log) == 0:
        return 1.0, 1.3, 1.3  # league-average fallback
    
    log['pts'] = np.where(log['gf'] > log['ga'], 3, np.where(log['gf'] == log['ga'], 1, 0))
    return log['pts'].mean(), log['gf'].mean(), log['ga'].mean()


def predict_match(home_team, away_team, elo_df, results_df, model, le, feature_cols,
                  home_is_host=0, away_is_host=0, is_knockout=0):
    """Predict Win/Draw/Loss probabilities for a matchup."""
    
    # Elo
    home_elo = get_team_elo(home_team, elo_df)
    away_elo = get_team_elo(away_team, elo_df)
    if home_elo is None or away_elo is None:
        return None  # can't predict without Elo
    
    # Form
    h_pts, h_gf, h_ga = get_team_form(home_team, results_df)
    a_pts, a_gf, a_ga = get_team_form(away_team, results_df)
    
    # Build the feature row in EXACT order the model expects
    features = pd.DataFrame([{
        'elo_diff': home_elo - away_elo,
        'home_elo': home_elo,
        'away_elo': away_elo,
        'home_form_pts': h_pts,
        'away_form_pts': a_pts,
        'form_pts_diff': h_pts - a_pts,
        'home_form_gf': h_gf,
        'away_form_gf': a_gf,
        'home_form_ga': h_ga,
        'away_form_ga': a_ga,
        'home_is_host': home_is_host,
        'away_is_host': away_is_host,
        'host_diff': home_is_host - away_is_host,
        'is_knockout': is_knockout
    }])[feature_cols]  # reorder to match training
    
    # Predict probabilities
    probs = model.predict_proba(features)[0]
    
    # Map back to readable labels
    result = {le.classes_[i]: round(float(probs[i]), 3) for i in range(len(probs))}
    return result


# --- TEST: load historical results for form, then predict a real matchup ---
results_hist = pd.read_csv('data/raw/historical/results.csv')
results_hist['date'] = pd.to_datetime(results_hist['date'])

print("Argentina (home) vs Brazil (away):")
print(predict_match('Argentina', 'Brazil', elo, results_hist, model, le, feature_cols))

Argentina (home) vs Brazil (away):
{'Away Win': 0.145, 'Draw': 0.323, 'Home Win': 0.532}


In [4]:
print("Argentina home vs Brazil away:")
print(predict_match('Argentina', 'Brazil', elo, results_hist, model, le, feature_cols))

print("\nBrazil home vs Argentina away:")
print(predict_match('Brazil', 'Argentina', elo, results_hist, model, le, feature_cols))

Argentina home vs Brazil away:
{'Away Win': 0.145, 'Draw': 0.323, 'Home Win': 0.532}

Brazil home vs Argentina away:
{'Away Win': 0.267, 'Draw': 0.272, 'Home Win': 0.461}


In [5]:
# --- World Cup-aware prediction: neutralize home bias, preserve host advantage ---

def predict_wc_match(team_a, team_b, elo_df, results_df, model, le, feature_cols,
                     host_team=None):
    """
    Predict a World Cup matchup.
    - If neither team is the host: neutral venue → average both orderings to cancel home bias.
    - If one team is the host: they get legitimate host advantage → predict with them as home.
    
    host_team: pass the host nation's name if this match is played in their country, else None.
    """
    
    # CASE 1: A host is playing in their own country → real advantage, no averaging
    if host_team == team_a:
        return predict_match(team_a, team_b, elo_df, results_df, model, le, feature_cols,
                             home_is_host=1, away_is_host=0)
    if host_team == team_b:
        # Host is team_b, so put them in the home slot where the advantage applies
        flipped = predict_match(team_b, team_a, elo_df, results_df, model, le, feature_cols,
                                home_is_host=1, away_is_host=0)
        # Relabel from team_b's perspective back to team_a's perspective
        return {
            'Home Win': flipped['Away Win'],   # team_a winning = away win in flipped
            'Draw': flipped['Draw'],
            'Away Win': flipped['Home Win']
        }
    
    # CASE 2: Neutral venue (no host) → average both orderings to remove home bias
    p1 = predict_match(team_a, team_b, elo_df, results_df, model, le, feature_cols)
    p2 = predict_match(team_b, team_a, elo_df, results_df, model, le, feature_cols)
    
    # p2 is from team_b's perspective, so flip it to team_a's perspective before averaging
    p2_flipped = {'Home Win': p2['Away Win'], 'Draw': p2['Draw'], 'Away Win': p2['Home Win']}
    
    return {
        'Home Win': round((p1['Home Win'] + p2_flipped['Home Win']) / 2, 3),
        'Draw':     round((p1['Draw'] + p2_flipped['Draw']) / 2, 3),
        'Away Win': round((p1['Away Win'] + p2_flipped['Away Win']) / 2, 3)
    }


# --- TEST: Argentina vs Brazil at a NEUTRAL venue (no host) ---
print("Argentina vs Brazil (neutral venue):")
print(predict_wc_match('Argentina', 'Brazil', elo, results_hist, model, le, feature_cols))

# --- TEST: Mexico vs Argentina, played in Mexico (host advantage) ---
print("\nMexico vs Argentina (Mexico hosting):")
print(predict_wc_match('Mexico', 'Argentina', elo, results_hist, model, le, feature_cols, host_team='Mexico'))

Argentina vs Brazil (neutral venue):
{'Home Win': 0.4, 'Draw': 0.297, 'Away Win': 0.303}

Mexico vs Argentina (Mexico hosting):
{'Away Win': 0.415, 'Draw': 0.229, 'Home Win': 0.356}


## Layer 1 (Group Stage Simulation)

Simulate the 12 groups of 4. Each team plays 3 group matches; results determine standings and who advances to the knockout round.

In [6]:
# --- Inspect group structure in the 2026 data ---
print("Columns available:", matches_2026.columns.tolist())
print("\nStage names:", matches_2026['stage_name'].unique())

# Look for group info — check if there's a group column or if it's in stage_name
print("\nSample of matches with teams:")
print(matches_2026[['home_team_name', 'away_team_name', 'stage_name']].head(10).to_string(index=False))

# Do we have the 48 teams and their groups anywhere?
print("\nUnique home teams:", matches_2026['home_team_name'].nunique())
print("Total unique teams (home+away):", 
      len(set(matches_2026['home_team_name']) | set(matches_2026['away_team_name'])))

Columns available: ['match_id', 'date', 'kickoff_time_utc', 'stage_name', 'stadium_name', 'city', 'country', 'home_team_name', 'home_fifa_code', 'away_team_name', 'away_fifa_code', 'home_score', 'away_score', 'status', 'home_xg', 'away_xg', 'home_goalkeeper', 'away_goalkeeper', 'player_of_the_match_name', 'referee_name']

Stage names: <StringArray>
['Group Stage']
Length: 1, dtype: str

Sample of matches with teams:
home_team_name         away_team_name  stage_name
        Mexico           South Africa Group Stage
   South Korea                Czechia Group Stage
        Canada Bosnia and Herzegovina Group Stage
           USA               Paraguay Group Stage
         Qatar            Switzerland Group Stage
        Brazil                Morocco Group Stage
         Haiti               Scotland Group Stage
     Australia                Türkiye Group Stage
       Germany                Curaçao Group Stage
   Netherlands                  Japan Group Stage

Unique home teams: 48
Total u

In [7]:
# --- Can we derive groups from the schedule? ---
# In group play, each team plays exactly 3 matches, all within their group.

from collections import defaultdict

# Count how many group matches each team plays
team_match_count = defaultdict(int)
for _, row in matches_2026.iterrows():
    team_match_count[row['home_team_name']] += 1
    team_match_count[row['away_team_name']] += 1

counts = pd.Series(team_match_count)
print("Matches played per team (group stage):")
print(counts.value_counts())
print("\nIf most teams show '3', group structure is cleanly derivable.")
print("\nSample:")
print(counts.sort_values(ascending=False).head(10))

Matches played per team (group stage):
3    48
Name: count, dtype: int64

If most teams show '3', group structure is cleanly derivable.

Sample:
Mexico          3
South Africa    3
Belgium         3
Egypt           3
Saudi Arabia    3
Uruguay         3
IR Iran         3
New Zealand     3
France          3
Senegal         3
dtype: int64


In [8]:
# --- Check 2026 team names against Elo names (catch mismatches early) ---
teams_2026 = set(matches_2026['home_team_name']) | set(matches_2026['away_team_name'])
elo_2026_teams = set(elo[elo['year'] == 2026]['country'].unique())

# Which 2026 match-data names don't exist in the Elo data?
mismatches = teams_2026 - elo_2026_teams
print("2026 team names NOT matching Elo names:")
for t in sorted(mismatches):
    print("  ", t)
print(f"\n{len(mismatches)} mismatches to fix")

# Show what Elo calls its teams, for comparison
print("\nFor reference — Elo's team names:")
print(sorted(elo_2026_teams))

2026 team names NOT matching Elo names:
   Cabo Verde
   Congo DR
   Côte d'Ivoire
   IR Iran
   Türkiye
   USA

6 mismatches to fix

For reference — Elo's team names:
['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curaçao', 'Czechia', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']


In [9]:
# --- Standardize 2026 match-data team names to Elo names ---
TEAM_NAME_FIX_2026 = {
    'Cabo Verde':     'Cape Verde',
    'Congo DR':       'DR Congo',
    "Côte d'Ivoire":  'Ivory Coast',
    'IR Iran':        'Iran',
    'Türkiye':        'Turkey',
    'USA':            'United States',
}

# Apply to both team columns in the 2026 match data
matches_2026['home_team_name'] = matches_2026['home_team_name'].replace(TEAM_NAME_FIX_2026)
matches_2026['away_team_name'] = matches_2026['away_team_name'].replace(TEAM_NAME_FIX_2026)

# --- Verify: re-check for mismatches ---
teams_2026 = set(matches_2026['home_team_name']) | set(matches_2026['away_team_name'])
elo_2026_teams = set(elo[elo['year'] == 2026]['country'].unique())
remaining = teams_2026 - elo_2026_teams

print("Remaining mismatches after fix:", remaining)
print("Should be empty set() →", len(remaining), "remaining")

Remaining mismatches after fix: set()
Should be empty set() → 0 remaining


In [10]:
# --- Layer 1a: Reconstruct groups from the match schedule ---
import networkx as nx

# Build a graph: each team is a node, each group match is an edge
G = nx.Graph()
for _, row in matches_2026.iterrows():
    G.add_edge(row['home_team_name'], row['away_team_name'])

# Connected components = the groups (teams only play within their group)
groups_raw = list(nx.connected_components(G))

print(f"Number of groups found: {len(groups_raw)}")
print(f"Teams per group: {[len(g) for g in groups_raw]}")
print("\nReconstructed groups:")
for i, group in enumerate(groups_raw):
    letter = chr(65 + i)  # A, B, C...
    print(f"  Group {letter}: {sorted(group)}")

Number of groups found: 12
Teams per group: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]

Reconstructed groups:
  Group A: ['Czechia', 'Mexico', 'South Africa', 'South Korea']
  Group B: ['Bosnia and Herzegovina', 'Canada', 'Qatar', 'Switzerland']
  Group C: ['Australia', 'Paraguay', 'Turkey', 'United States']
  Group D: ['Brazil', 'Haiti', 'Morocco', 'Scotland']
  Group E: ['Curaçao', 'Ecuador', 'Germany', 'Ivory Coast']
  Group F: ['Japan', 'Netherlands', 'Sweden', 'Tunisia']
  Group G: ['Cape Verde', 'Saudi Arabia', 'Spain', 'Uruguay']
  Group H: ['Belgium', 'Egypt', 'Iran', 'New Zealand']
  Group I: ['France', 'Iraq', 'Norway', 'Senegal']
  Group J: ['Algeria', 'Argentina', 'Austria', 'Jordan']
  Group K: ['Colombia', 'DR Congo', 'Portugal', 'Uzbekistan']
  Group L: ['Croatia', 'England', 'Ghana', 'Panama']


In [16]:
# --- Layer 1b: Simulate a single group's standings (FIXED) ---
from itertools import combinations

def simulate_group(group_teams, elo_df, results_df, model, le, feature_cols):
    """Simulate all matches in a group, return standings sorted by points."""
    points = {team: 0 for team in group_teams}

    for team_a, team_b in combinations(group_teams, 2):
        probs = predict_wc_match(team_a, team_b, elo_df, results_df, model, le, feature_cols)

        # Normalize so probabilities sum to EXACTLY 1.0 (rounding can make them 0.999/1.001)
        p = np.array([probs['Home Win'], probs['Draw'], probs['Away Win']])
        p = p / p.sum()

        outcome = np.random.choice(['Home Win', 'Draw', 'Away Win'], p=p)

        if outcome == 'Home Win':
            points[team_a] += 3
        elif outcome == 'Away Win':
            points[team_b] += 3
        else:
            points[team_a] += 1
            points[team_b] += 1

    standings = sorted(group_teams, key=lambda t: (points[t], np.random.random()), reverse=True)
    return standings, points


# --- TEST on Argentina's group ---
test_group = list(groups_raw[9])
standings, points = simulate_group(test_group, elo, results_hist, model, le, feature_cols)

print("Group J simulation (one run):")
for rank, team in enumerate(standings, 1):
    status = "✓ ADVANCES" if rank <= 2 else ""
    print(f"  {rank}. {team:25s} {points[team]} pts  {status}")

Group J simulation (one run):
  1. Algeria                   9 pts  ✓ ADVANCES
  2. Argentina                 6 pts  ✓ ADVANCES
  3. Austria                   3 pts  
  4. Jordan                    0 pts  


In [17]:
# --- Layer 1c: Simulate ALL 12 groups, collect qualifiers ---

def simulate_all_groups(groups_list, elo_df, results_df, model, le, feature_cols):
    """Simulate every group. Return group winners, runners-up, and third-place teams."""
    winners = []        # 1st place (12 teams)
    runners_up = []     # 2nd place (12 teams)
    third_place = []    # 3rd place candidates (12 teams, only best 8 advance)
    
    for group in groups_list:
        standings, points = simulate_group(list(group), elo_df, results_df, model, le, feature_cols)
        winners.append((standings[0], points[standings[0]]))
        runners_up.append((standings[1], points[standings[1]]))
        third_place.append((standings[2], points[standings[2]]))
    
    return winners, runners_up, third_place


# --- Test one full group-stage simulation ---
winners, runners_up, third_place = simulate_all_groups(
    groups_raw, elo, results_hist, model, le, feature_cols
)

print("GROUP WINNERS (12):")
for team, pts in winners:
    print(f"  {team:25s} {pts} pts")

print("\nRUNNERS-UP (12):")
for team, pts in runners_up:
    print(f"  {team:25s} {pts} pts")

print("\nTHIRD PLACE (best 8 advance):")
third_sorted = sorted(third_place, key=lambda x: (x[1], np.random.random()), reverse=True)
for i, (team, pts) in enumerate(third_sorted):
    status = "✓" if i < 8 else "✗ OUT"
    print(f"  {team:25s} {pts} pts  {status}")

GROUP WINNERS (12):
  Mexico                    7 pts
  Canada                    6 pts
  Turkey                    6 pts
  Brazil                    7 pts
  Ecuador                   7 pts
  Netherlands               6 pts
  Cape Verde                7 pts
  New Zealand               6 pts
  France                    5 pts
  Algeria                   6 pts
  Portugal                  9 pts
  England                   7 pts

RUNNERS-UP (12):
  Czechia                   4 pts
  Switzerland               6 pts
  Australia                 6 pts
  Scotland                  6 pts
  Germany                   3 pts
  Japan                     6 pts
  Spain                     6 pts
  Belgium                   6 pts
  Senegal                   5 pts
  Argentina                 5 pts
  Colombia                  6 pts
  Panama                    4 pts

THIRD PLACE (best 8 advance):
  Bosnia and Herzegovina    6 pts  ✓
  Jordan                    4 pts  ✓
  Norway                    4 pts  ✓
  Tu

In [18]:
# --- Layer 1b (Fork 2): Group sim with head-to-head points tiebreaker ---
from itertools import combinations

def simulate_group(group_teams, elo_df, results_df, model, le, feature_cols):
    """Simulate a group. Tiebreak on head-to-head points, then random (documented approximation)."""
    teams = list(group_teams)
    points = {t: 0 for t in teams}
    # Track results so we can compute head-to-head if teams tie
    match_results = {}  # (team_a, team_b) -> 'a' / 'b' / 'draw'

    for team_a, team_b in combinations(teams, 2):
        probs = predict_wc_match(team_a, team_b, elo_df, results_df, model, le, feature_cols)
        p = np.array([probs['Home Win'], probs['Draw'], probs['Away Win']])
        p = p / p.sum()
        outcome = np.random.choice(['Home Win', 'Draw', 'Away Win'], p=p)

        if outcome == 'Home Win':
            points[team_a] += 3
            match_results[(team_a, team_b)] = 'a'
        elif outcome == 'Away Win':
            points[team_b] += 3
            match_results[(team_a, team_b)] = 'b'
        else:
            points[team_a] += 1
            points[team_b] += 1
            match_results[(team_a, team_b)] = 'draw'

    # --- Head-to-head points among tied teams ---
    def h2h_points(team, tied_group):
        """Points this team earned only against others in the tied set."""
        pts = 0
        for other in tied_group:
            if other == team:
                continue
            key = (team, other) if (team, other) in match_results else (other, team)
            res = match_results.get(key)
            if res is None:
                continue
            if key == (team, other):
                if res == 'a': pts += 3
                elif res == 'draw': pts += 1
            else:  # key == (other, team)
                if res == 'b': pts += 3
                elif res == 'draw': pts += 1
        return pts

    # Sort: primary = total points, secondary = head-to-head points, tertiary = random
    def sort_key(team):
        tied = [t for t in teams if points[t] == points[team]]
        h2h = h2h_points(team, tied) if len(tied) > 1 else 0
        return (points[team], h2h, np.random.random())

    standings = sorted(teams, key=sort_key, reverse=True)
    return standings, points


# --- Test on Argentina's group ---
test_group = list(groups_raw[9])
standings, points = simulate_group(test_group, elo, results_hist, model, le, feature_cols)
print("Group J (head-to-head tiebreaker active):")
for rank, team in enumerate(standings, 1):
    status = "✓ ADVANCES" if rank <= 2 else ""
    print(f"  {rank}. {team:25s} {points[team]} pts  {status}")

Group J (head-to-head tiebreaker active):
  1. Austria                   7 pts  ✓ ADVANCES
  2. Argentina                 7 pts  ✓ ADVANCES
  3. Algeria                   3 pts  
  4. Jordan                    0 pts  


In [21]:
# Re-run all-groups with the upgraded tiebreaker logic
winners, runners_up, third_place = simulate_all_groups(
    groups_raw, elo, results_hist, model, le, feature_cols
)

# Collect all 32 qualifiers cleanly
third_sorted = sorted(third_place, key=lambda x: (x[1], np.random.random()), reverse=True)
qualifiers = (
    [t for t, p in winners] +
    [t for t, p in runners_up] +
    [t for t, p in third_sorted[:8]]
)

print("Total qualifiers for Round of 32:", len(qualifiers))
print("\nShould be exactly 32 →", len(qualifiers) == 32)

Total qualifiers for Round of 32: 32

Should be exactly 32 → True


## Layer 2 (Knockout Bracket)

Construct the Round of 32 → Final using a seeded single-elimination bracket (simplified from FIFA's exact third-place slotting). Knockout matches use `is_knockout=1` and must produce a winner, no draws, so ties are resolved by re-sampling (representing extra time / penalties).

**Bracket flow:** Round of 32 → Round of 16 → Quarter-finals → Semi-finals → Final → Champion.

**Simplification note:** Teams are seeded into the bracket by group-stage performance rather than FIFA's official third-place mapping table. This produces realistic championship probabilities while keeping the bracket logic clean. Exact FIFA slotting is a possible v2 enhancement.

In [22]:
# --- Layer 2a: A single knockout match (no draws allowed) ---

def play_knockout_match(team_a, team_b, elo_df, results_df, model, le, feature_cols,
                        host_team=None):
    """
    Play one knockout match. Must produce a winner.
    If the model samples a Draw, re-distribute that probability between the two
    win outcomes (represents extra time / penalties resolving the tie).
    Returns the winning team.
    """
    probs = predict_wc_match(team_a, team_b, elo_df, results_df, model, le, feature_cols,
                             host_team=host_team)

    home_win = probs['Home Win']
    away_win = probs['Away Win']
    draw = probs['Draw']

    # Split the draw probability proportionally between the two teams' win chances
    # (a stronger team is more likely to win the shootout/extra time too)
    total_win = home_win + away_win
    if total_win == 0:
        # extreme edge case — fall back to 50/50
        home_final, away_final = 0.5, 0.5
    else:
        home_final = home_win + draw * (home_win / total_win)
        away_final = away_win + draw * (away_win / total_win)

    # Normalize to be safe
    p = np.array([home_final, away_final])
    p = p / p.sum()

    winner = np.random.choice([team_a, team_b], p=p)
    return winner


# --- Test: play the same knockout match 1000 times, see the win split ---
from collections import Counter

results_count = Counter()
for _ in range(1000):
    w = play_knockout_match('Argentina', 'Brazil', elo, results_hist, model, le, feature_cols)
    results_count[w] += 1

print("Argentina vs Brazil — 1000 knockout simulations:")
print(f"  Argentina advances: {results_count['Argentina']} times ({results_count['Argentina']/10:.1f}%)")
print(f"  Brazil advances:    {results_count['Brazil']} times ({results_count['Brazil']/10:.1f}%)")
print("\n(No draws possible — every match produces a winner)")

Argentina vs Brazil — 1000 knockout simulations:
  Argentina advances: 592 times (59.2%)
  Brazil advances:    408 times (40.8%)

(No draws possible — every match produces a winner)


In [23]:
# --- Layer 2b: Play a full knockout bracket from 32 teams to champion ---

def play_knockout_bracket(qualifiers, elo_df, results_df, model, le, feature_cols, verbose=False):
    """
    Take 32 qualifiers, play single-elimination down to a champion.
    Returns the champion and (optionally) the round-by-round results.
    """
    round_names = {32: 'Round of 32', 16: 'Round of 16', 8: 'Quarter-finals',
                   4: 'Semi-finals', 2: 'Final'}

    current_round = list(qualifiers)  # seeded order: strongest to weakest
    bracket_history = {}

    while len(current_round) > 1:
        round_name = round_names.get(len(current_round), f'Round of {len(current_round)}')
        winners = []

        # Pair teams: 1st vs last, 2nd vs 2nd-last, etc. (standard seeding)
        # current_round is already seeded, so pair top half against bottom half
        n = len(current_round)
        for i in range(n // 2):
            team_a = current_round[i]
            team_b = current_round[n - 1 - i]
            winner = play_knockout_match(team_a, team_b, elo_df, results_df,
                                          model, le, feature_cols)
            winners.append(winner)

        bracket_history[round_name] = winners
        if verbose:
            print(f"{round_name}: {len(winners)} advance")
        current_round = winners

    champion = current_round[0]
    return champion, bracket_history


# --- Test: one full bracket run ---
champion, history = play_knockout_bracket(
    qualifiers, elo, results_hist, model, le, feature_cols, verbose=True
)

print(f"\n🏆 CHAMPION: {champion}")
print("\nFinalists:", history['Final'])
print("Semi-finalists:", history['Semi-finals'])

Round of 32: 16 advance
Round of 16: 8 advance
Quarter-finals: 4 advance
Semi-finals: 2 advance
Final: 1 advance

🏆 CHAMPION: Brazil

Finalists: [np.str_('Brazil')]
Semi-finalists: [np.str_('Brazil'), np.str_('Argentina')]


## Layer 3 (Monte Carlo Loop)

Run the full tournament (groups → knockouts → champion) 10,000+ times. Count how often each team wins to produce championship probabilities. 

**True Monte Carlo**: outcomes are randomly sampled from match probabilities, not deterministically assigned to favorites.

In [25]:
# --- Optimization: cache predictions so each matchup is computed only once ---

# Build a cache of all possible matchup probabilities up front
prediction_cache = {}

def predict_cached(team_a, team_b, elo_df, results_df, model, le, feature_cols):
    """Return cached neutral-venue probabilities, computing only on first request."""
    key = (team_a, team_b)
    if key not in prediction_cache:
        prediction_cache[key] = predict_wc_match(team_a, team_b, elo_df, results_df,
                                                  model, le, feature_cols)
    return prediction_cache[key]

# Pre-warm the cache: compute every possible matchup among the 48 teams ONCE
all_teams = sorted(set(elo[elo['year'] == 2026]['country'].unique()))
print(f"Pre-computing all matchups among {len(all_teams)} teams...")

import time
start = time.time()
count = 0
for i, ta in enumerate(all_teams):
    for tb in all_teams:
        if ta != tb:
            predict_cached(ta, tb, elo, results_hist, model, le, feature_cols)
            count += 1
print(f"Cached {count} matchups in {time.time()-start:.1f}s")
print(f"Cache size: {len(prediction_cache)} entries")

Pre-computing all matchups among 48 teams...
Cached 2256 matchups in 67.4s
Cache size: 2256 entries


In [26]:
# --- Cache-powered versions of the simulation functions ---

def simulate_group_fast(group_teams, elo_df, results_df, model, le, feature_cols):
    teams = list(group_teams)
    points = {t: 0 for t in teams}
    match_results = {}

    for team_a, team_b in combinations(teams, 2):
        probs = predict_cached(team_a, team_b, elo_df, results_df, model, le, feature_cols)
        p = np.array([probs['Home Win'], probs['Draw'], probs['Away Win']])
        p = p / p.sum()
        outcome = np.random.choice(['Home Win', 'Draw', 'Away Win'], p=p)
        if outcome == 'Home Win':
            points[team_a] += 3; match_results[(team_a, team_b)] = 'a'
        elif outcome == 'Away Win':
            points[team_b] += 3; match_results[(team_a, team_b)] = 'b'
        else:
            points[team_a] += 1; points[team_b] += 1; match_results[(team_a, team_b)] = 'draw'

    def h2h_points(team, tied_group):
        pts = 0
        for other in tied_group:
            if other == team: continue
            key = (team, other) if (team, other) in match_results else (other, team)
            res = match_results.get(key)
            if res is None: continue
            if key == (team, other):
                if res == 'a': pts += 3
                elif res == 'draw': pts += 1
            else:
                if res == 'b': pts += 3
                elif res == 'draw': pts += 1
        return pts

    def sort_key(team):
        tied = [t for t in teams if points[t] == points[team]]
        h2h = h2h_points(team, tied) if len(tied) > 1 else 0
        return (points[team], h2h, np.random.random())

    return sorted(teams, key=sort_key, reverse=True), points


def play_knockout_match_fast(team_a, team_b, elo_df, results_df, model, le, feature_cols):
    probs = predict_cached(team_a, team_b, elo_df, results_df, model, le, feature_cols)
    hw, aw, dr = probs['Home Win'], probs['Away Win'], probs['Draw']
    total = hw + aw
    if total == 0:
        p = np.array([0.5, 0.5])
    else:
        p = np.array([hw + dr*(hw/total), aw + dr*(aw/total)])
        p = p / p.sum()
    return np.random.choice([team_a, team_b], p=p)


def simulate_all_groups_fast(groups_list, elo_df, results_df, model, le, feature_cols):
    winners, runners_up, third_place = [], [], []
    for group in groups_list:
        standings, points = simulate_group_fast(list(group), elo_df, results_df, model, le, feature_cols)
        winners.append((standings[0], points[standings[0]]))
        runners_up.append((standings[1], points[standings[1]]))
        third_place.append((standings[2], points[standings[2]]))
    return winners, runners_up, third_place


def play_knockout_bracket_fast(qualifiers, elo_df, results_df, model, le, feature_cols):
    round_names = {32:'Round of 32',16:'Round of 16',8:'Quarter-finals',4:'Semi-finals',2:'Final'}
    current = list(qualifiers)
    history = {}
    while len(current) > 1:
        rn = round_names.get(len(current), f'Round of {len(current)}')
        n = len(current)
        winners = [play_knockout_match_fast(current[i], current[n-1-i], elo_df, results_df,
                                            model, le, feature_cols) for i in range(n//2)]
        history[rn] = winners
        current = winners
    return current[0], history


# --- Re-time 100 simulations with caching ---
def run_monte_carlo_fast(n, groups_list, elo_df, results_df, model, le, feature_cols):
    champions, finalists, semifinalists = Counter(), Counter(), Counter()
    start = time.time()
    for i in range(n):
        w, r, t = simulate_all_groups_fast(groups_list, elo_df, results_df, model, le, feature_cols)
        t_sorted = sorted(t, key=lambda x: (x[1], np.random.random()), reverse=True)
        quals = [x for x,_ in w] + [x for x,_ in r] + [x for x,_ in t_sorted[:8]]
        champ, hist = play_knockout_bracket_fast(quals, elo_df, results_df, model, le, feature_cols)
        champions[str(champ)] += 1
        for f in hist['Final']: finalists[str(f)] += 1
        for s in hist['Semi-finals']: semifinalists[str(s)] += 1
    print(f"{n} simulations in {time.time()-start:.2f}s")
    return champions, finalists, semifinalists

champions, finalists, semifinalists = run_monte_carlo_fast(
    100, groups_raw, elo, results_hist, model, le, feature_cols)

100 simulations in 0.23s


In [27]:
# --- THE REAL RUN: 10,000 simulations ---
print("Running 10,000 simulations...\n")
champions, finalists, semifinalists = run_monte_carlo_fast(
    10000, groups_raw, elo, results_hist, model, le, feature_cols)

print("\n" + "="*45)
print("🏆 WORLD CUP 2026 — CHAMPIONSHIP ODDS")
print("="*45)
for team, count in champions.most_common(20):
    pct = count / 10000 * 100
    bar = "█" * int(pct)
    print(f"  {team:22s} {pct:5.1f}%  {bar}")

Running 10,000 simulations...

10000 simulations in 12.83s

🏆 WORLD CUP 2026 — CHAMPIONSHIP ODDS
  Argentina               20.1%  ████████████████████
  France                  15.0%  ██████████████
  Brazil                  11.3%  ███████████
  England                  8.1%  ████████
  Spain                    6.9%  ██████
  Portugal                 5.7%  █████
  Colombia                 4.5%  ████
  Netherlands              3.4%  ███
  Norway                   3.4%  ███
  Switzerland              2.9%  ██
  Germany                  2.4%  ██
  Belgium                  2.3%  ██
  Japan                    2.2%  ██
  Ecuador                  2.1%  ██
  Mexico                   1.5%  █
  Turkey                   1.4%  █
  Croatia                  1.2%  █
  Uruguay                  1.1%  █
  Senegal                  0.9%  
  Canada                   0.8%  
